# RAVDESS Preprocessing Script
This script is used for generating RAVDESS_RAW_PREPROCESSED data from the raw RAW data, you can download the raw dataset from: https://zenodo.org/record/1188976

emotion: (neutral、calm、happy、sad、angry、fearful、disgust、surprised)

In [1]:
import torch
from facenet_pytorch import MTCNN
import os, sys
import glob
import pickle
import numpy as np
import pandas as pd
import cv2
from scipy.io import wavfile
from tqdm import tqdm
import torch
from facenet_pytorch import MTCNN

In [2]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
mtcnn = MTCNN(image_size=48, margin=2, post_process=False, device=device)

# Common Functions

In [3]:
def read_video(file_name):
    vidcap = cv2.VideoCapture(file_name)
    
    # Read FPS
    (major_ver, minor_ver, subminor_ver) = (cv2.__version__).split('.')
    if int(major_ver)  < 3 :
        fps = vidcap.get(cv2.cv.CV_CAP_PROP_FPS)
    else :
        fps = vidcap.get(cv2.CAP_PROP_FPS)
    
    # Read image data
    success, image = vidcap.read()
    images = []
    while success:
        images.append(image)
        success, image = vidcap.read()
    return np.stack(images), fps

def dump_image(img_segment, out_path='./'):
    count = 0
    for i in range(img_segment.shape[0]):
        faces = mtcnn(img_segment[i,:,:,:])
        if faces != None:
#             cv2.imwrite(f'{out_path}/image_{i}.jpg', img_segment[i,:,:,:])
            cv2.imwrite(f'{out_path}/image_{count}.jpg', faces.permute(1, 2, 0).int().numpy())
            count = count + 1
#             mtcnn(img_segment[i,:,:,:], save_path=f'{out_path}/image_{i}.jpg')

In [4]:
%%time
# Process multimodal data over all sessions
# NOTE: This might take several hours to run, the time listed on this cell is for processing 5 label files
output_path = r'/home/matt/Model/Database_processed/RAVDESS_RAW_PROCESSED_Face'
#     list wav file
wav_path = r'/home/matt/Model/Data/RAVDESS/audio_file'
wav_list = os.listdir(wav_path)
for i in range(len(wav_list)):
    wav_list[i] = wav_list[i][:-4]
if not os.path.exists(output_path):
    os.makedirs(output_path)
    
all_metas = {}
for base_path in glob.glob(r'/home/matt/Model/Data/RAVDESS/Video_Song/*'):
    for root, dirs, files in os.walk(base_path, topdown=False):
        for name in files:
            print(os.path.join(root, name))
            if name[-3:] == 'mp4':
#                 print('1', os.path.join(root, name))
                audio_file_name = name[:-3] + 'wav'
#                 print('2', audio_file_name)
                index = wav_list.index(audio_file_name[:-4])
                sr, signal  = wavfile.read(os.path.join(wav_path, wav_list[index]) + '.wav')
                images, fps = read_video(os.path.join(root, name))
#                 print('3', os.path.join(wav_path, wav_list[index]) + '.wav')
            out_path = os.path.join(output_path, audio_file_name[:-4])
#             print(out_path)
            if not os.path.exists(out_path):
                os.makedirs(out_path)
            wavfile.write(f'{out_path}/audio.wav', sr, signal)
            dump_image(images, out_path)
for base_path in glob.glob(r'/home/matt/Model/Data/RAVDESS/Video_Speech/*'):
    for root, dirs, files in os.walk(base_path, topdown=False):
        for name in files:
            print(os.path.join(root, name))
            if name[-3:] == 'mp4':
#                 print('1', os.path.join(root, name))
                audio_file_name = name[:-3] + 'wav'
#                 print('2', audio_file_name)
                index = wav_list.index(audio_file_name[:-4])
                sr, signal  = wavfile.read(os.path.join(wav_path, wav_list[index]) + '.wav')
                images, fps = read_video(os.path.join(root, name))
#                 print('3', os.path.join(wav_path, wav_list[index]) + '.wav')
            out_path = os.path.join(output_path, audio_file_name[:-4])
#             print(out_path)
            if not os.path.exists(out_path):
                os.makedirs(out_path)
            wavfile.write(f'{out_path}/audio.wav', sr, signal)
            dump_image(images, out_path)

/home/matt/Model/Data/RAVDESS/Video_Song/Video_Song_Actor_19/Actor_19/02-02-02-02-01-02-19.mp4
/home/matt/Model/Data/RAVDESS/Video_Song/Video_Song_Actor_19/Actor_19/02-02-05-01-02-02-19.mp4
/home/matt/Model/Data/RAVDESS/Video_Song/Video_Song_Actor_19/Actor_19/02-02-04-01-01-02-19.mp4
/home/matt/Model/Data/RAVDESS/Video_Song/Video_Song_Actor_19/Actor_19/01-02-04-02-01-02-19.mp4
/home/matt/Model/Data/RAVDESS/Video_Song/Video_Song_Actor_19/Actor_19/02-02-06-02-01-02-19.mp4
/home/matt/Model/Data/RAVDESS/Video_Song/Video_Song_Actor_19/Actor_19/02-02-01-01-01-01-19.mp4
/home/matt/Model/Data/RAVDESS/Video_Song/Video_Song_Actor_19/Actor_19/02-02-06-01-02-02-19.mp4
/home/matt/Model/Data/RAVDESS/Video_Song/Video_Song_Actor_19/Actor_19/01-02-05-02-02-01-19.mp4
/home/matt/Model/Data/RAVDESS/Video_Song/Video_Song_Actor_19/Actor_19/01-02-05-01-02-02-19.mp4
/home/matt/Model/Data/RAVDESS/Video_Song/Video_Song_Actor_19/Actor_19/02-02-04-02-02-01-19.mp4
/home/matt/Model/Data/RAVDESS/Video_Song/Video_Son

In [5]:
df = pd.read_csv('/home/matt/Model/Data/RAVDESS/statistic.csv')
df

,filename,emotion,duration,width,height,text
0,01-02-01-01-01-01-01.mp4,neutral,4.29,1280,720,Kids are talking by the door
1,01-02-01-01-01-02-01.mp4,neutral,4.31,1280,720,Kids are talking by the door
2,01-02-01-01-02-01-01.mp4,neutral,4.22,1280,720,Dogs are sitting by the door
3,01-02-01-01-02-02-01.mp4,neutral,4.18,1280,720,Dogs are sitting by the door
4,01-02-02-01-01-01-01.mp4,calm,4.42,1280,720,Kids are talking by the door
...,...,...,...,...,...,...
4899,02-01-08-01-02-02-24.mp4,surprised,3.46,1280,720,Dogs are sitting by the door
4900,02-01-08-02-01-01-24.mp4,surprised,3.99,1280,720,Kids are talking by the door
4901,02-01-08-02-01-02-24.mp4,surprised,4.01,1280,720,Kids are talking by the door
4902,02-01-08-02-02-01-24.mp4,surprised,3.71,1280,720,Dogs are sitting by the door


In [6]:
metadata = {}
# emo_trans = {'01': 'neutral', '02': 'calm', '03': 'happy', '04': 'sad', '05': 'angry', '06': 'fearful', '07': 'disgust', '08': 'surprised'}
emo_trans = {'01': 'neu', '02': 'cal', '03': 'hap', '04': 'sad', '05': 'ang', '06': 'fea', '07': 'dis', '08': 'sur'}
df = pd.read_csv('/home/matt/Model/Data/RAVDESS/statistic.csv')
for index, row in df.iterrows():
    print(row['filename'])
    print(row['text'])
    emo = row['filename'].split('-')[2]
    metadata[row['filename'][:-4]] = {'text': row['text'], 'label': emo_trans[emo]}

pickle.dump(metadata, open(r'/home/matt/Model/Database_processed/RAVDESS_RAW_PROCESSED_Face/meta.pkl','wb'))

01-02-01-01-01-01-01.mp4
Kids are talking by the door
01-02-01-01-01-02-01.mp4
Kids are talking by the door
01-02-01-01-02-01-01.mp4
Dogs are sitting by the door
01-02-01-01-02-02-01.mp4
Dogs are sitting by the door
01-02-02-01-01-01-01.mp4
Kids are talking by the door
01-02-02-01-01-02-01.mp4
Kids are talking by the door
01-02-02-01-02-01-01.mp4
Dogs are sitting by the door
01-02-02-01-02-02-01.mp4
Dogs are sitting by the door
01-02-02-02-01-01-01.mp4
Kids are talking by the door
01-02-02-02-01-02-01.mp4
Kids are talking by the door
01-02-02-02-02-01-01.mp4
Dogs are sitting by the door
01-02-02-02-02-02-01.mp4
Dogs are sitting by the door
01-02-03-01-01-01-01.mp4
Kids are talking by the door
01-02-03-01-01-02-01.mp4
Kids are talking by the door
01-02-03-01-02-01-01.mp4
Dogs are sitting by the door
01-02-03-01-02-02-01.mp4
Dogs are sitting by the door
01-02-03-02-01-01-01.mp4
Kids are talking by the door
01-02-03-02-01-02-01.mp4
Kids are talking by the door
01-02-03-02-02-01-01.mp4
Dog

In [7]:
import pickle
with open('/home/matt/Model/Database_processed/RAVDESS_RAW_PROCESSED_Face/meta.pkl', 'rb') as f:
    data = pickle.load(f)

In [8]:
data

{'01-02-01-01-01-01-01': {'text': 'Kids are talking by the door',
  'label': 'neu'},
 '01-02-01-01-01-02-01': {'text': 'Kids are talking by the door',
  'label': 'neu'},
 '01-02-01-01-02-01-01': {'text': 'Dogs are sitting by the door',
  'label': 'neu'},
 '01-02-01-01-02-02-01': {'text': 'Dogs are sitting by the door',
  'label': 'neu'},
 '01-02-02-01-01-01-01': {'text': 'Kids are talking by the door',
  'label': 'cal'},
 '01-02-02-01-01-02-01': {'text': 'Kids are talking by the door',
  'label': 'cal'},
 '01-02-02-01-02-01-01': {'text': 'Dogs are sitting by the door',
  'label': 'cal'},
 '01-02-02-01-02-02-01': {'text': 'Dogs are sitting by the door',
  'label': 'cal'},
 '01-02-02-02-01-01-01': {'text': 'Kids are talking by the door',
  'label': 'cal'},
 '01-02-02-02-01-02-01': {'text': 'Kids are talking by the door',
  'label': 'cal'},
 '01-02-02-02-02-01-01': {'text': 'Dogs are sitting by the door',
  'label': 'cal'},
 '01-02-02-02-02-02-01': {'text': 'Dogs are sitting by the door',